# 5.8 · 梯度提升分类 / Gradient Boosting (GBDT) Classifier

> **课程定位 / Where this fits**
> 5.7 的随机森林是 **bagging**(并行、独立、降方差)。GBDT 是 **boosting**(串行、每棵树纠正前面的错、降偏差)。4.13 讲过回归版的梯度提升; 这里是分类版, 并把它当作 5.9–5.11 三巨头(XGBoost/LightGBM/CatBoost)的**数学地基**。
> GBDT trains trees sequentially, each correcting the previous error via functional gradient descent. The math foundation for the boosting big-three.

> 💡 **面试相关 / Interview-relevant**
> - "boosting 与 bagging 的本质区别" ★★★★★
> - "梯度提升的'梯度'是对什么求导" ★★★★★（对预测值/函数空间）
> - "GBDT 分类如何工作(拟合 log-odds 残差)" ★★★★
> - "学习率(shrinkage)的作用" ★★★★★
> - "GBDT 为什么用浅树(弱学习器)" ★★★★
> - "GBDT 怎么防过拟合" ★★★★

---

## 学习目标 / Learning Objectives
1. boosting 思想 + 与 bagging 的对立。
2. **函数空间的梯度下降**: 每棵树拟合损失的负梯度(伪残差)。
3. 分类版: 在 **log-odds** 空间加法建模。
4. 学习率 × 树数的权衡 + 早停。
5. 对照 sklearn `GradientBoosting` / `HistGradientBoosting`。

## 目录 / TOC
1. [boosting vs bagging ⭐](#1)
2. [函数梯度下降 ⭐](#2)
3. [💰 数据: 合成 Adult Income](#3)
4. [从零: 二分类 GBDT ⭐](#4)
5. [学习率 × 树数 + 早停 ⭐](#5)
6. [sklearn + HistGBDT](#6)
7. [小结](#7)


<a id="1"></a>
## 1. boosting vs bagging ⭐ / Boosting vs Bagging

| | Bagging (RF, 5.7) | Boosting (GBDT) |
|---|---|---|
| 树关系 | **独立并行** | **串行依赖**, 后树修前树的错 |
| 基学习器 | 深树(低偏差高方差) | **浅树**(高偏差低方差, 弱学习器) |
| 主攻 | 降**方差** | 降**偏差** |
| 加树过多 | 不过拟合 | **会**过拟合 |

**boosting 直觉**: 先用一个弱模型, 看它哪里错了, 再训一个新模型**专门补这个错**, 加权叠加, 反复。无数弱学习器叠成强模型。


<a id="2"></a>
## 2. 函数空间的梯度下降 ⭐ / Functional Gradient Descent

GBDT 把模型写成加法形式 $F_M(\mathbf{x})=\sum_{m=1}^M \nu\, h_m(\mathbf{x})$, 逐步加树。第 $m$ 步, 我们想减小损失 $L(y, F)$。

**关键洞察**: 把 $F(\mathbf{x})$ 当作"参数", 对它求梯度。损失下降最快的方向是**负梯度**:
$$r_{im} = -\left[\frac{\partial L(y_i, F(\mathbf{x}_i))}{\partial F(\mathbf{x}_i)}\right]_{F=F_{m-1}}$$

这就是**伪残差(pseudo-residual)**。然后训一棵树 $h_m$ **拟合这些伪残差**, 再以学习率 $\nu$ 加进去。

- **回归 + 平方损失**(4.13): 伪残差 $= y_i - F(\mathbf{x}_i)$, 就是**普通残差**。
- **分类 + 对数损失**: $F$ 是 **log-odds**, 伪残差 $= y_i - \sigma(F(\mathbf{x}_i)) = y_i - p_i$ —— 又是 $(y-p)$! 和 5.1/5.2 同一个量。

所以分类 GBDT = **在 log-odds 空间反复拟合 $(y-p)$ 残差**。


<a id="3"></a>
## 3. 数据: 合成 Adult Income / Synthetic Adult Income

真 UCI Adult(成年人收入>50K 预测)需下载。这里**内联合成**一个同风格数据集: 用年龄、教育年限、每周工时、资本利得等特征, 按一个含交互项的真实规律生成"高收入"标签。5.8–5.11 都用它, 方便横向对比四种 boosting。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

def make_income(n=8000, seed=0):
    rng = np.random.default_rng(seed)
    age = rng.integers(18, 70, n)
    edu_years = rng.integers(6, 21, n)
    hours = rng.normal(40, 10, n).clip(10, 80)
    capital_gain = (rng.random(n) < 0.15) * rng.exponential(5000, n)
    # 真实 log-odds 规律: 含非线性(年龄平方) + 交互(教育×工时)
    logit = (-9 + 0.04*age - 0.0004*(age-45)**2 + 0.25*edu_years
             + 0.02*hours + 0.0002*np.sqrt(capital_gain)*edu_years*0.3
             + rng.normal(0, 0.5, n))
    y = (rng.random(n) < 1/(1+np.exp(-logit))).astype(int)
    X = pd.DataFrame({"age": age, "edu_years": edu_years, "hours": hours,
                      "capital_gain": capital_gain.round(0)})
    return X, y

X, y = make_income()
print(f"合成 Adult Income: {X.shape}, 高收入率 {y.mean():.0%}")
print(X.describe().round(1).to_string())

from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X.values, y, test_size=0.3, stratify=y, random_state=0)


<a id="4"></a>
## 4. 从零: 二分类 GBDT ⭐ / From Scratch

按第 2 节的公式实现: 从 log-odds 常数起步, 每轮用浅回归树拟合伪残差 $(y-p)$, 学习率加进去。


In [ ]:
from sklearn.tree import DecisionTreeRegressor

def sigmoid(z): return 1/(1+np.exp(-np.clip(z, -500, 500)))

class GBDTClassifier:
    def __init__(self, n_trees=100, lr=0.1, max_depth=3):
        self.n_trees, self.lr, self.max_depth = n_trees, lr, max_depth
    def fit(self, X, y):
        self.F0 = np.log(y.mean()/(1-y.mean()))   # 初始 log-odds = 全局先验
        F = np.full(len(y), self.F0)
        self.trees = []
        for _ in range(self.n_trees):
            p = sigmoid(F)
            residual = y - p                        # 伪残差 = y - p (对数损失)
            tree = DecisionTreeRegressor(max_depth=self.max_depth).fit(X, residual)
            F += self.lr * tree.predict(X)
            self.trees.append(tree)
        return self
    def decision(self, X):
        F = np.full(len(X), self.F0)
        for t in self.trees: F += self.lr * t.predict(X)
        return F
    def predict_proba(self, X): return sigmoid(self.decision(X))
    def predict(self, X): return (self.predict_proba(X) > 0.5).astype(int)

gb = GBDTClassifier(n_trees=100, lr=0.1, max_depth=3).fit(X_tr, y_tr)
from sklearn.metrics import accuracy_score, roc_auc_score
print(f"从零 GBDT test 准确率: {accuracy_score(y_te, gb.predict(X_te)):.3f}")
print(f"从零 GBDT test AUC:    {roc_auc_score(y_te, gb.predict_proba(X_te)):.3f}")


In [ ]:
# 对照 sklearn / vs sklearn
from sklearn.ensemble import GradientBoostingClassifier
sk = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0).fit(X_tr, y_tr)
print(f"sklearn GBDT test 准确率: {sk.score(X_te, y_te):.3f}")
print(f"sklearn GBDT test AUC:    {roc_auc_score(y_te, sk.predict_proba(X_te)[:,1]):.3f}")
print("从零实现与库结果接近 → 验证了'拟合(y-p)伪残差'的核心机制")


<a id="5"></a>
## 5. 学习率 × 树数 + 早停 ⭐ / Learning Rate × Trees & Early Stopping

**学习率 $\nu$(shrinkage)**: 每棵树只迈一小步。小学习率 → 需更多树, 但泛化更好(类似 SGD 小步长)。**经验法则**: 小 $\nu$(0.01–0.1)+ 多树 + **早停**。这与 RF 相反——GBDT **树太多会过拟合**。


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for lr in [0.01, 0.1, 0.5]:
    aucs = []
    for nt in [10, 30, 60, 100, 150, 200]:
        m = GradientBoostingClassifier(n_estimators=nt, learning_rate=lr, max_depth=3, random_state=0).fit(X_tr, y_tr)
        aucs.append(roc_auc_score(y_te, m.predict_proba(X_te)[:,1]))
    ax.plot([10,30,60,100,150,200], aucs, "o-", label=f"lr={lr}")
ax.set_xlabel("n_estimators"); ax.set_ylabel("test AUC"); ax.legend()
ax.set_title("学习率×树数: 大lr 快但易过拟合; 小lr 需更多树更稳")
plt.tight_layout(); plt.show()

# 早停 / early stopping via validation
es = GradientBoostingClassifier(n_estimators=500, learning_rate=0.1, max_depth=3,
                                validation_fraction=0.2, n_iter_no_change=10, random_state=0).fit(X_tr, y_tr)
print(f"早停: 设上限500棵, 实际只用 {es.n_estimators_} 棵就停了 (验证集不再提升)")
print(f"早停模型 test AUC: {roc_auc_score(y_te, es.predict_proba(X_te)[:,1]):.3f}")


<a id="6"></a>
## 6. sklearn + HistGBDT / Histogram-based GBDT

sklearn 的 `HistGradientBoostingClassifier` 把连续特征**分箱(直方图)**, 大幅加速(借鉴 LightGBM, 见 5.10), 大数据上比经典 `GradientBoosting` 快几个数量级, 还原生支持缺失值。


In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
import time
for name, model in [("GradientBoosting", GradientBoostingClassifier(n_estimators=200, random_state=0)),
                    ("HistGradientBoosting", HistGradientBoostingClassifier(max_iter=200, random_state=0))]:
    t = time.perf_counter(); model.fit(X_tr, y_tr); dt = time.perf_counter()-t
    auc = roc_auc_score(y_te, model.predict_proba(X_te)[:,1])
    print(f"{name:<22} 训练 {dt:.2f}s  test AUC {auc:.3f}")
print("注: 仅 8000 行时分箱的固定开销没赚回来, Hist 不一定更快;")
print("    它的优势在'大数据(数十万行+)'时才显著——那时分箱让每次分裂从 O(n) 降到 O(bins)。")
print("HistGBDT 是现代 boosting(5.9-5.11)的思路源头, 大数据优先。")


<a id="7"></a>
## 7. 小结 / Summary

```
boosting: 串行, 每棵浅树纠正前面的错, 降偏差 (vs bagging 并行降方差)
函数梯度下降: 树拟合损失的负梯度(伪残差)
  回归+平方损失: 残差 = y - F
  分类+对数损失: 伪残差 = y - σ(F) = y - p (log-odds 空间), 又是 (y-p)
学习率 ν: 小步长更稳, 需更多树; GBDT 树太多会过拟合 → 早停
HistGBDT: 特征分箱加速, 现代 boosting 起点
```

### 💡 面试速查
1. **boosting 串行降偏差**(浅弱树), bagging 并行降方差(深树)
2. **梯度**是对**预测值 F(x)** 求导(函数空间), 树拟合负梯度=伪残差
3. **分类伪残差 = y - p**(在 log-odds 空间加法建模)
4. **学习率小 + 树多 + 早停**; 树过多会过拟合(与 RF 不同)
5. **HistGBDT** 分箱加速, 是 LightGBM 思想的体现

### 下一节
**5.9 XGBoost**——在 GBDT 上加二阶泰勒近似、正则化叶子权重、列采样等工程与数学改进, Kaggle 表格赛长期霸主。
